# 14 · Temporal Feature

Single notebook that builds all three temporal outputs from raw MusicBrainz sources.
Run top-to-bottom to fully rebuild.

## Outputs

| File | Description |
|---|---|
| `data/features/album_era.parquet` | Per-album: `album_id`, `best_year`, `best_year_source`, `era_bin` |
| `data/features/album_era_matrix.npz` | One-hot era matrix, `n_albums × N_ERA_COLS` |
| `data/features/album_temporal_matrix.npz` | Era one-hot \| year continuous (app block) |
| `data/features/temporal_year_scaler.json` | `year_min` / `year_max` for the year column |

## Inputs

`data/mb_album.parquet`, `data/mb_release_year.parquet`, `data/mb_album_country.parquet`,
`data/mb_album_artists.parquet`, `data/mb_artist.parquet`, `data/features/album_ids.pkl`

## Pipeline

```
RAW SOURCES
  mb_release_year     → release_group_meta_year  (best source)
  mb_album_country    → album_country_year        (fallback 1)
  mb_artist           → artist_begin_year         (fallback 2, noisiest)
        │
        ▼  combine_first fallback chain
  best_year + best_year_source + era_bin
        │
        ├──▶ album_era.parquet          (audit / downstream joins)
        ├──▶ album_era_matrix.npz       (one-hot, 10 cols)
        └──▶ album_temporal_matrix.npz  (era × 1.0 ∥ year × 0.3, N+1 cols)
```

## Era bins

| Bin | Years | Notes |
|---|---|---|
| `Pre-1900` | < 1900 | 50-yr bucket — very sparse |
| `1900–1949` | 1900–1949 | 50-yr bucket — sparse pre-modern |
| `1950s` – `2020s` | decade bins | standard 10-yr buckets |
| `Unknown` | — | no year in any source — **zero row, no column** |

## Why YEAR_WEIGHT = 0.3

The year column is min-max scaled over the full `best_year` range (~975–2026, span ≈ 1051 years).
A decade is only 10/1051 ≈ 0.95% of that range — **intra-decade differentiation in cosine
space is effectively zero** regardless of weight (Δcosine < 0.00001 for same-era pairs).

The year column's only meaningful effect is **era-boundary smoothing**: albums released just
before/after a decade turn (e.g. Dec 1969 / Jan 1970) have zero era cosine (different bins)
but get a gentle non-zero signal from the shared year proximity.

At `YEAR_WEIGHT = 0.3`, a boundary pair gets temporal cosine ≈ 0.10. With the Era dial at
default (dial 4, W_temporal ≈ 0.73), that contributes ≈ 0.04 to the full cosine numerator —
a nudge, not a signal. Genre and label similarity dominate completely.
Higher weights aggressively bridge genuinely different eras (year range too compressed to
distinguish 1969/1970 from 1989/2000).

In [ ]:
import pickle
import json
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz, load_npz, hstack

DATA_DIR     = '../data'
FEATURES_DIR = f'{DATA_DIR}/features'

# ── Internal weights baked into album_temporal_matrix.npz ─────────────────
# The Era dial in the app multiplies the whole block uniformly on top of these.
# Change YEAR_WEIGHT here then re-run the full notebook to rebuild all outputs.
ERA_WEIGHT  = 1.0   # era one-hot columns — always 1.0
YEAR_WEIGHT = 0.3   # year continuous column — see rationale in title cell

# ── Known year corrections ────────────────────────────────────────────────
# Five albums with confirmed bad years, verified via web search during EDA.
# Applied AFTER the fallback chain so raw source columns remain auditable.
# best_year_source is set to 'manual_correction' for these rows.
#
# | album_id | Album                              | Raw  | Corrected | Evidence              |
# |----------|------------------------------------|------|-----------|-----------------------|
# | 4576816  | Signes & Racines (Adag'nan)        | 2027 | 2021      | Bandcamp / Amazon     |
# | 4598782  | Original Music Box Melodies Xmas   | 2069 | 1969      | Discogs (Pickwick)    |
# | 4615228  | Meditations Vol. 1 (Helisir)       | 2205 | 2025      | Bandcamp              |
# | 4643359  | diSTILLed (Russ Still)             | 2202 | 2022      | Transposition error   |
# | 4706113  | The Entertainment (The Clockworks) | 2027 | 2026      | V2 Records (Mar 2026) |
YEAR_CORRECTIONS = {
    4576816: 2021,
    4598782: 1969,
    4615228: 2025,
    4643359: 2022,
    4706113: 2026,
}

# Era bin vocabulary — Unknown excluded (zero row, not a column)
ERA_ORDER = ['Pre-1900', '1900–1949'] + [f'{d}s' for d in range(1950, 2030, 10)]

print(f'ERA_WEIGHT={ERA_WEIGHT}  YEAR_WEIGHT={YEAR_WEIGHT}')
print(f'Era bins ({len(ERA_ORDER)}): {ERA_ORDER}')

## 1 · Load raw sources

Four files provide the year signals. Each is a different path into MusicBrainz:

- **`mb_release_year`** — `release_group_meta.first_release_date_year`. The canonical
  first-release year per release group. Most reliable; only rows with a non-NULL year
  are included (albums with no date are absent, not zero).
- **`mb_album_country`** — earliest country-specific release date per album. Used as
  fallback 1; available for albums whose release group has no canonical date but whose
  individual country releases do.
- **`mb_artist`** — artist formation / birth year (`begin_date_year`). Fallback 2 and
  the noisiest source: a 2005 comeback album by an artist formed in 1968 gets year 1968
  from this source, which is wrong. Accepted as a last resort.
- **`mb_album`** — scope boundary. Every album in `valid_albums` appears here; the
  merge starts from this list so zero-year albums are preserved as NULL rows.

In [ ]:
albums = (
    pd.read_parquet(f'{DATA_DIR}/mb_album.parquet')
    .rename(columns={'id': 'album_id'})
)

rg_year = pd.read_parquet(f'{DATA_DIR}/mb_release_year.parquet')

country_year = (
    pd.read_parquet(
        f'{DATA_DIR}/mb_album_country.parquet',
        columns=['album_id', 'album_year'],
    )
    .rename(columns={'album_year': 'album_country_year'})
)

album_artists = pd.read_parquet(
    f'{DATA_DIR}/mb_album_artists.parquet',
    columns=['album_id', 'artist_id', 'artist_name'],
)
artists = (
    pd.read_parquet(
        f'{DATA_DIR}/mb_artist.parquet',
        columns=['id', 'artist_year'],
    )
    .rename(columns={'id': 'artist_id', 'artist_year': 'artist_begin_year'})
)
artist_year = (
    album_artists
    .merge(artists, on='artist_id', how='left')
    [['album_id', 'artist_name', 'artist_begin_year']]
)

print(f'Albums in scope  : {len(albums):,}')
print(f'rg_year rows     : {len(rg_year):,}')
print(f'country_year rows: {len(country_year):,}')
print(f'artist_year rows : {len(artist_year):,}')

## 2 · Assemble + fallback chain

Left-joins all three year sources onto the master album list (scope boundary = `mb_album`).
Every album appears in the output even if all three year columns are NULL.

Fallback chain (`combine_first` applies priority order):
```
best_year = release_group_meta_year
         ?? album_country_year
         ?? artist_begin_year
```

`best_year_source` records which source won — useful for auditing. Assignments from
`artist_begin` are lowest-confidence and most likely to be wrong for comeback/reissue albums.

In [ ]:
df = (
    albums[['album_id', 'name']]
    .merge(rg_year,       on='album_id', how='left')
    .merge(country_year,  on='album_id', how='left')
    .merge(artist_year,   on='album_id', how='left')
)

# Fallback chain
df['best_year'] = (
    df['release_group_meta_year']
    .combine_first(df['album_country_year'])
    .combine_first(df['artist_begin_year'])
)
df['best_year_source'] = np.select(
    [
        df['release_group_meta_year'].notna(),
        df['album_country_year'].notna(),
        df['artist_begin_year'].notna(),
    ],
    ['release_group_meta', 'album_country', 'artist_begin'],
    default='unknown',
)

print(f'Assembled: {len(df):,} rows')
print('\nSource breakdown (before corrections):')
print(df['best_year_source'].value_counts().to_string())

In [ ]:
# Apply manual corrections
print('Applying year corrections:')
for album_id, corrected_year in YEAR_CORRECTIONS.items():
    mask = df['album_id'] == album_id
    old  = df.loc[mask, 'best_year'].values
    df.loc[mask, 'best_year']        = float(corrected_year)
    df.loc[mask, 'best_year_source'] = 'manual_correction'
    old_str = f'{old[0]:.0f}' if len(old) and not pd.isna(old[0]) else 'NaN'
    print(f'  album_id={album_id}  {old_str} → {corrected_year}')

In [ ]:
# Assign era bins
def assign_era(year):
    if pd.isna(year):  return 'Unknown'
    y = int(year)
    if y > 2026:       return 'Unknown'   # hard cap: future / erroneous dates
    if y < 1900:       return 'Pre-1900'
    if y < 1950:       return '1900–1949'
    return f'{(y // 10) * 10}s'

df['era_bin'] = df['best_year'].apply(assign_era)

era_order_full = ERA_ORDER + ['Unknown']
print('Era bin counts:')
print(
    df['era_bin']
    .value_counts()
    .reindex(era_order_full, fill_value=0)
    .to_string()
)
n_unknown = (df['era_bin'] == 'Unknown').sum()
print(f'\nUnknown: {n_unknown:,} ({n_unknown / len(df) * 100:.2f}%)')

## 3 · Save `album_era.parquet`

The intermediate parquet keeps the raw year and source columns for audit and downstream
joins (e.g. displaying the era in app results, checking which albums use `artist_begin`
as their year source). It is the input to the year column in the temporal matrix — using
the same `best_year` here keeps both sub-signals internally consistent.

In [ ]:
era_out = df[['album_id', 'best_year', 'best_year_source', 'era_bin']]
era_out.to_parquet(
    f'{FEATURES_DIR}/album_era.parquet', index=False, compression='zstd'
)
print(f'Saved: album_era.parquet  ({era_out.shape[0]:,} rows)')
print(f'  Unknown era : {(era_out["era_bin"] == "Unknown").sum():,}')
print(f'  Known era   : {(era_out["era_bin"] != "Unknown").sum():,}')

## 4 · Build `album_era_matrix.npz`

Sparse one-hot matrix aligned to `album_ids.pkl`:
- **Rows** = albums in exact `album_ids.pkl` order
- **Columns** = era bins in chronological order (`ERA_ORDER` — `Unknown` excluded)
- **Values** = 1.0 (float32)

**Why `Unknown` has no column:** two no-data albums would appear similar to each other —
a false signal. Zero rows correctly represent "no era information".

This matrix is kept as a standalone file so the EDA notebook and any other downstream
code can load the era signal without the year column attached.

In [ ]:
with open(f'{FEATURES_DIR}/album_ids.pkl', 'rb') as f:
    album_ids = pickle.load(f)

album_index = pd.Index(album_ids)
n_albums    = len(album_index)
print(f'Master album universe: {n_albums:,}')

In [ ]:
era_index = pd.Index(ERA_ORDER)
n_eras    = len(era_index)

era_known = era_out[era_out['era_bin'] != 'Unknown'].copy()

row_idx = album_index.get_indexer(era_known['album_id'].values)
col_idx = era_index.get_indexer(era_known['era_bin'].values)
valid   = (row_idx >= 0) & (col_idx >= 0)

X_era = csr_matrix(
    (
        np.ones(valid.sum(), dtype=np.float32),
        (row_idx[valid], col_idx[valid]),
    ),
    shape=(n_albums, n_eras),
)

print(f'X_era shape   : {X_era.shape}')
print(f'nnz           : {X_era.nnz:,}')
print(f'Coverage      : {(X_era.getnnz(axis=1) > 0).sum() / n_albums * 100:.1f}%')
print(f'Zero rows     : {(X_era.getnnz(axis=1) == 0).sum():,}  (Unknown era)')

In [ ]:
# Spot-check: column sums should match era_bin value counts
col_sums = np.asarray(X_era.sum(axis=0)).flatten()
print('Albums per era column:')
for era, count in zip(ERA_ORDER, col_sums):
    print(f'  {era:<12} {int(count):>9,}')

In [ ]:
save_npz(f'{FEATURES_DIR}/album_era_matrix.npz', X_era)
print(f'Saved: album_era_matrix.npz  {X_era.shape}')

## 5 · Build year column

Reads `best_year` from `album_era.parquet` (just saved) and builds a single-column
sparse matrix scaled to [0, 1]. Using the same `best_year` source as the era one-hot
keeps both sub-signals internally consistent — the same album_id gets the same year
value in both.

Unknown-era albums get zero in the year column (consistent with their all-zero era row).
Albums that have an era bin but a NULL `best_year` (shouldn't exist after the fallback
chain, but guarded defensively) also get zero.

**Why the year column has limited intra-decade power:** the data spans ~975–2026 (1051 yrs).
A decade is <1% of that range in scaled space, so same-era cosine changes by < 0.00001
regardless of weight. The column's only real effect is era-boundary smoothing — see title
cell for the full analysis and rationale for YEAR_WEIGHT = 0.3.

In [ ]:
# Align era parquet to album_ids row order
era_aligned = (
    era_out
    .set_index('album_id')
    .reindex(album_ids)
    [['best_year', 'era_bin']]
    .reset_index(drop=True)
)

is_unknown = (
    era_aligned['era_bin'].isna() |
    (era_aligned['era_bin'] == 'Unknown')
)
is_known = ~is_unknown & era_aligned['best_year'].notna()

known_years = era_aligned.loc[is_known, 'best_year']
year_min    = float(known_years.min())
year_max    = float(known_years.max())
print(f'Year range (known albums): {year_min:.0f} – {year_max:.0f}  (span={year_max-year_min:.0f} yrs)')
print(f'Known   : {is_known.sum():,} ({is_known.sum() / n_albums * 100:.1f}%)')
print(f'Unknown : {is_unknown.sum():,} ({is_unknown.sum() / n_albums * 100:.2f}%)  → year_col = 0')

# Save scaler params
with open(f'{FEATURES_DIR}/temporal_year_scaler.json', 'w') as f:
    json.dump({'year_min': year_min, 'year_max': year_max}, f, indent=2)
print('Saved: temporal_year_scaler.json')

# Build dense array, then convert to sparse
year_arr = np.zeros(n_albums, dtype=np.float32)
year_arr[is_known.values] = (
    (known_years.values - year_min) / (year_max - year_min)
).astype(np.float32)

nonzero_rows = np.where(year_arr > 0)[0].astype(np.int32)
year_col_csr = csr_matrix(
    (
        year_arr[nonzero_rows],
        (nonzero_rows, np.zeros(len(nonzero_rows), dtype=np.int32)),
    ),
    shape=(n_albums, 1),
    dtype=np.float32,
)
print(f'Year column: {year_col_csr.shape}  nnz={year_col_csr.nnz:,}')

## 6 · Build `album_temporal_matrix.npz`

Horizontal stack of the era one-hot block and the year column:

```
columns 0 … N_ERA-1  :  era one-hot  × ERA_WEIGHT  (1.0)
column  N_ERA        :  year scaled  × YEAR_WEIGHT  (0.3)
```

This is the matrix loaded by the app as the `era` block. The Era dial in the app
multiplies the **entire** block uniformly — the internal ERA_WEIGHT : YEAR_WEIGHT ratio
is fixed at build time and controls only the relative balance between the two sub-signals.

In [ ]:
N_ERA_COLS      = X_era.shape[1]
N_TEMPORAL_COLS = N_ERA_COLS + 1

X_temporal = hstack(
    [
        X_era.multiply(ERA_WEIGHT),
        year_col_csr.multiply(YEAR_WEIGHT),
    ],
    format='csr',
).astype(np.float32)

print(f'Temporal matrix  : {X_temporal.shape}')
print(f'  era cols (0–{N_ERA_COLS-1}) : weight={ERA_WEIGHT}')
print(f'  year col ({N_ERA_COLS})     : weight={YEAR_WEIGHT}')
print(f'  nnz              : {X_temporal.nnz:,}')
print(f'  albums with signal: {(X_temporal.getnnz(axis=1) > 0).sum():,} '
      f'({(X_temporal.getnnz(axis=1) > 0).sum() / n_albums * 100:.1f}%)')

In [ ]:
# ── Sanity checks ─────────────────────────────────────────────────────────
assert X_temporal.shape == (n_albums, N_TEMPORAL_COLS), \
    f'Shape: got {X_temporal.shape}, expected ({n_albums}, {N_TEMPORAL_COLS})'

# Every album with an era row must still have signal in the temporal matrix
era_signal  = X_era.getnnz(axis=1) > 0
temp_signal = X_temporal.getnnz(axis=1) > 0
assert (temp_signal >= era_signal).all(), \
    'Some era-covered albums lost signal in temporal matrix'

# Year column nnz preserved after hstack
year_nnz_check = int(X_temporal.tocsc()[:, N_ERA_COLS].nnz)
assert year_nnz_check == year_col_csr.nnz, \
    f'Year column nnz mismatch: {year_nnz_check} vs {year_col_csr.nnz}'

print('Sanity checks passed ✓')
print(f'  Era cols nnz : {X_era.nnz:,}')
print(f'  Year col nnz : {year_nnz_check:,}')
print(f'  Combined nnz : {X_temporal.nnz:,}')

In [ ]:
save_npz(f'{FEATURES_DIR}/album_temporal_matrix.npz', X_temporal)

print('─' * 60)
print('All outputs saved:')
print(f'  album_era.parquet          {era_out.shape[0]:,} rows')
print(f'  album_era_matrix.npz       {X_era.shape}  nnz={X_era.nnz:,}')
print(f'  temporal_year_scaler.json  year_min={year_min:.0f}  year_max={year_max:.0f}')
print(f'  album_temporal_matrix.npz  {X_temporal.shape}  nnz={X_temporal.nnz:,}')
print()
print('Year column effect at YEAR_WEIGHT=0.3 (Era dial=4, W_temporal≈0.73):')
print('  Intra-decade Δcosine  : < 0.00001  (year range too wide for resolution)')
print('  Era-boundary cosine   : ≈ 0.10     (1969/1970-type pairs)')
print('  Full-cosine nudge     : ≈ 0.04     (genre + label still dominate)')